# The Manual Training Loop

Every PyTorch training loop is the same five steps:
1. Forward pass
2. Compute loss
3. Zero gradients
4. Backward pass
5. Update weights

This notebook builds it from scratch so you understand what higher-level frameworks hide.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_classification
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## Toy dataset

In [ ]:
X_np, y_np = make_classification(n_samples=1000, n_features=20, random_state=42)

X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32)

split = int(0.8 * len(X))
train_ds = TensorDataset(X[:split], y[:split])
val_ds   = TensorDataset(X[split:], y[split:])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)

## Model

In [ ]:
model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1)
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()   # combines sigmoid + BCE, more numerically stable
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## The Five-Step Loop

In [ ]:
for epoch in range(10):

    # --- TRAINING PHASE ---
    model.train()   # enables Dropout, BatchNorm running stats, etc.
    train_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        preds = model(xb).squeeze()          # 1. forward
        loss  = criterion(preds, yb)         # 2. loss

        optimizer.zero_grad()                # 3. zero grads  (must come before backward)
        loss.backward()                      # 4. backward
        optimizer.step()                     # 5. update

        train_loss += loss.item()

    # --- VALIDATION PHASE ---
    model.eval()    # disables Dropout, freezes BatchNorm stats
    val_loss = correct = total = 0

    with torch.no_grad():   # no gradient tracking needed for eval
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            preds = model(xb).squeeze()
            val_loss += criterion(preds, yb).item()
            correct  += ((preds > 0).float() == yb).sum().item()
            total    += yb.size(0)

    print(f"Epoch {epoch+1:2d}  "
          f"train_loss={train_loss/len(train_loader):.4f}  "
          f"val_loss={val_loss/len(val_loader):.4f}  "
          f"val_acc={correct/total:.4f}")

## Why zero_grad() before backward?

PyTorch **accumulates** gradients by default. Without `zero_grad()`, each `backward()` adds to whatever gradients are already stored — after two batches, `param.grad` holds the sum of both batches' gradients, not just the current one.

Call `optimizer.zero_grad()` before every `loss.backward()` to get clean per-batch gradients.

In [ ]:
# Demonstration: grad accumulation without zero_grad
p = torch.tensor(2.0, requires_grad=True)
for i in range(3):
    loss = p ** 2
    loss.backward()
    print(f"After backward {i+1}: p.grad = {p.grad}")  # 4, 8, 12 — keeps adding

## model.train() vs model.eval()

| Mode | Dropout | BatchNorm |
|---|---|---|
| `model.train()` | active (randomly zeroes units) | uses batch stats |
| `model.eval()` | off (all units pass through) | uses running stats |

Forgetting `model.eval()` during validation gives noisy results because Dropout randomly drops outputs even at inference time.

In [ ]:
# Demonstrate: same input, different outputs in train mode due to Dropout
dropout_model = nn.Sequential(nn.Linear(20, 64), nn.Dropout(0.5), nn.Linear(64, 1))
x = torch.ones(1, 20)

dropout_model.train()
print("train:", dropout_model(x).item(), dropout_model(x).item())  # different each call

dropout_model.eval()
print("eval: ", dropout_model(x).item(), dropout_model(x).item())  # identical

## Saving and Loading a Checkpoint

In [ ]:
# Save — only the weights, not the model class
torch.save(model.state_dict(), 'checkpoint.pth')

# Load — must recreate the architecture first
loaded_model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1)
)
loaded_model.load_state_dict(torch.load('checkpoint.pth', map_location='cpu'))
loaded_model.eval()
print("Loaded successfully")